<a href="https://colab.research.google.com/github/bindu-arpula/satellite-fire-risk-analytics/blob/main/NASA_VIIRS_Project_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
file_path = r"/content/drive/MyDrive/NASA_FIRMS_TableauPrep_Cleaned.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Rows and coumns:", df.shape)
print(df.columns.tolist())

Dataset loaded successfully
Rows and coumns: (220679, 16)
['Response Priority', 'Fire Intensity Category', 'Day Night Label', 'Latitude', 'Longitude', 'Brightness I4', 'Scan', 'Track', 'Acquisition Date', 'Acquisition Time UTC', 'Satellite', 'Confidence', 'Version', 'Brightness I5', 'Fire Radiative Power', 'Day Night']


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

print("Column names after standardising:")
for col in df.columns:
    print("-", col)

Column names after standardising:
- Response_Priority
- Fire_Intensity_Category
- Day_Night_Label
- Latitude
- Longitude
- Brightness_I4
- Scan
- Track
- Acquisition_Date
- Acquisition_Time_UTC
- Satellite
- Confidence
- Version
- Brightness_I5
- Fire_Radiative_Power
- Day_Night


In [ ]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nFirst 5 rows:")
display(df.head())

Total rows: 220679
Total columns: 16

Missing values in each column:
Response_Priority          0
Fire_Intensity_Category    0
Day_Night_Label            0
Latitude                   0
Longitude                  0
Brightness_I4              0
Scan                       0
Track                      0
Acquisition_Date           0
Acquisition_Time_UTC       0
Satellite                  0
Confidence                 0
Version                    0
Brightness_I5              0
Fire_Radiative_Power       0
Day_Night                  0
dtype: int64

Duplicate rows:
0

First 5 rows:


,Response_Priority,Fire_Intensity_Category,Day_Night_Label,Latitude,Longitude,Brightness_I4,Scan,Track,Acquisition_Date,Acquisition_Time_UTC,Satellite,Confidence,Version,Brightness_I5,Fire_Radiative_Power,Day_Night
0,Low Priority,Low Intensity,Day,-15.86315,126.20374,330.44,0.56,0.52,05-05-2026,449,N,nominal,2.0NRT,297.82,8.95,D
1,Low Priority,Low Intensity,Day,44.45554,132.34802,334.06,0.61,0.53,05-05-2026,323,N,nominal,2.0NRT,295.77,6.27,D
2,Low Priority,Low Intensity,Day,-42.34604,146.56287,331.00,0.50,0.41,05-05-2026,440,N,nominal,2.0NRT,286.30,8.86,D
3,Low Priority,Low Intensity,Day,-15.86508,126.18790,334.19,0.56,0.52,05-05-2026,449,N,nominal,2.0NRT,292.90,8.09,D
4,Low Priority,Low Intensity,Day,-31.75750,148.89952,349.68,0.36,0.57,05-05-2026,442,N,nominal,2.0NRT,291.88,6.06,D


In [ ]:
df["Acquisition_Date"] = pd.to_datetime(
    df["Acquisition_Date"],
    errors="coerce",
    dayfirst=True
)

df["Acquisition_Time_UTC"] = (
    df["Acquisition_Time_UTC"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.extract(r"(\d+)", expand=False)
    .fillna("0000")
    .str.zfill(4)
)

df["Hour_UTC"] = df["Acquisition_Time_UTC"].str[:2].astype(int)
df["Minute_UTC"] = df["Acquisition_Time_UTC"].str[2:].astype(int)

df["Acquisition_Datetime_UTC"] = pd.to_datetime(
    df["Acquisition_Date"].dt.strftime("%Y-%m-%d") + " " +
    df["Hour_UTC"].astype(str).str.zfill(2) + ":" +
    df["Minute_UTC"].astype(str).str.zfill(2),
    errors="coerce"
)

df["Day_Name"] = df["Acquisition_Datetime_UTC"].dt.day_name()

print("Date and time conversion completed.")

display(
    df[[
        "Acquisition_Date",
        "Acquisition_Time_UTC",
        "Hour_UTC",
        "Minute_UTC",
        "Acquisition_Datetime_UTC",
        "Day_Name"
    ]].head(10)
)

Date and time conversion completed.


,Acquisition_Date,Acquisition_Time_UTC,Hour_UTC,Minute_UTC,Acquisition_Datetime_UTC,Day_Name
0,2026-05-05,0449,4,49,2026-05-05 04:49:00,Tuesday
1,2026-05-05,0323,3,23,2026-05-05 03:23:00,Tuesday
2,2026-05-05,0440,4,40,2026-05-05 04:40:00,Tuesday
3,2026-05-05,0449,4,49,2026-05-05 04:49:00,Tuesday
4,2026-05-05,0442,4,42,2026-05-05 04:42:00,Tuesday
5,2026-05-05,0323,3,23,2026-05-05 03:23:00,Tuesday
6,2026-05-05,0449,4,49,2026-05-05 04:49:00,Tuesday
7,2026-05-05,0440,4,40,2026-05-05 04:40:00,Tuesday
8,2026-05-05,0323,3,23,2026-05-05 03:23:00,Tuesday
9,2026-05-05,0442,4,42,2026-05-05 04:42:00,Tuesday


In [ ]:
print("Start date:", df["Acquisition_Date"].min())
print("End date:", df["Acquisition_Date"].max())

print("\nRecord count by date:")
display(df["Acquisition_Date"].value_counts().sort_index())

Start date: 2026-03-05 00:00:00
End date: 2026-10-05 00:00:00

Record count by date:


,count
Acquisition_Date,
2026-03-05,30678
2026-04-05,33174
2026-05-05,27445
2026-06-05,26388
2026-07-05,30369
2026-08-05,28289
2026-09-05,26705
2026-10-05,17631


In [ ]:
numeric_fields = [
    "Latitude",
    "Longitude",
    "Brightness_I4",
    "Brightness_I5",
    "Scan",
    "Track",
    "Fire_Radiative_Power"
]

for field in numeric_fields:
    if field in df.columns:
        df[field] = pd.to_numeric(df[field], errors="coerce")

print("Numeric conversion completed.")

print("\nData types after conversion:")
print(df[numeric_fields].dtypes)

Numeric conversion completed.

Data types after conversion:
Latitude                float64
Longitude               float64
Brightness_I4           float64
Brightness_I5           float64
Scan                    float64
Track                   float64
Fire_Radiative_Power    float64
dtype: object


In [ ]:
print("Day/Night label counts:")
print(df["Day_Night_Label"].value_counts())

print("\nOriginal Day_Night values:")
print(df["Day_Night"].value_counts())

Day/Night label counts:
Day_Night_Label
Day      164837
Night     55842
Name: count, dtype: int64

Original Day_Night values:
Day_Night
D    164837
N     55842
Name: count, dtype: int64


In [ ]:
print("Fire intensity category counts:")
print(df["Fire_Intensity_Category"].value_counts())

Fire intensity category counts:
Fire_Intensity_Category
Low Intensity         219810
Moderate Intensity       819
High Intensity            50
Name: count, dtype: int64


In [ ]:
print("Response priority counts:")
print(df["Response_Priority"].value_counts())

Response priority counts:
Response_Priority
Low Priority       220037
Medium Priority       621
High Priority          21
Name: count, dtype: int64


In [ ]:
df["Latitude_Grid"] = df["Latitude"].round(1)
df["Longitude_Grid"] = df["Longitude"].round(1)

df["Hotspot_Grid"] = (
    df["Latitude_Grid"].astype(str) + ", " + df["Longitude_Grid"].astype(str)
)

print("Hotspot grid created.")

display(
    df[[
        "Latitude",
        "Longitude",
        "Latitude_Grid",
        "Longitude_Grid",
        "Hotspot_Grid"
    ]].head(10)
)

Hotspot grid created.


,Latitude,Longitude,Latitude_Grid,Longitude_Grid,Hotspot_Grid
0,-15.86315,126.20374,-15.9,126.2,"-15.9, 126.2"
1,44.45554,132.34802,44.5,132.3,"44.5, 132.3"
2,-42.34604,146.56287,-42.3,146.6,"-42.3, 146.6"
3,-15.86508,126.18790,-15.9,126.2,"-15.9, 126.2"
4,-31.75750,148.89952,-31.8,148.9,"-31.8, 148.9"
5,43.38509,128.42474,43.4,128.4,"43.4, 128.4"
6,-15.85839,126.12204,-15.9,126.1,"-15.9, 126.1"
7,-42.33494,146.55873,-42.3,146.6,"-42.3, 146.6"
8,43.39077,128.42163,43.4,128.4,"43.4, 128.4"
9,-32.23719,145.98460,-32.2,146.0,"-32.2, 146.0"


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Total columns",
        "Start date",
        "End date",
        "Missing Latitude",
        "Missing Longitude",
        "Missing Fire Radiative Power",
        "Duplicate rows",
        "Average Fire Radiative Power",
        "Maximum Fire Radiative Power",
        "Low Priority count",
        "Medium Priority count",
        "High Priority count",
        "Day detections",
        "Night detections"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["Acquisition_Date"].min(),
        df["Acquisition_Date"].max(),
        df["Latitude"].isnull().sum(),
        df["Longitude"].isnull().sum(),
        df["Fire_Radiative_Power"].isnull().sum(),
        df.duplicated().sum(),
        round(df["Fire_Radiative_Power"].mean(), 2),
        round(df["Fire_Radiative_Power"].max(), 2),
        (df["Response_Priority"] == "Low Priority").sum(),
        (df["Response_Priority"] == "Medium Priority").sum(),
        (df["Response_Priority"] == "High Priority").sum(),
        (df["Day_Night_Label"] == "Day").sum(),
        (df["Day_Night_Label"] == "Night").sum()
    ]
})

display(summary)

,Metric,Value
0,Total rows,220679
1,Total columns,23
2,Start date,2026-03-05 00:00:00
3,End date,2026-10-05 00:00:00
4,Missing Latitude,0
5,Missing Longitude,0
6,Missing Fire Radiative Power,0
7,Duplicate rows,0
8,Average Fire Radiative Power,7.61
9,Maximum Fire Radiative Power,657.63


In [ ]:
import os

project_folder = "/content/drive/MyDrive/NASA_FIRMS_Tableau_Project"

# Create folder if it does not exist
os.makedirs(project_folder, exist_ok=True)

# Output file paths
final_tableau_file = project_folder + "/NASA_FIRMS_Final_For_Tableau.csv"
summary_file = project_folder + "/NASA_FIRMS_Data_Quality_Summary.csv"

# Export final files
df.to_csv(final_tableau_file, index=False)
summary.to_csv(summary_file, index=False)

print("Files exported successfully.")
print("Final Tableau dataset:", final_tableau_file)
print("Data quality summary:", summary_file)

Files exported successfully.
Final Tableau dataset: /content/drive/MyDrive/NASA_FIRMS_Tableau_Project/NASA_FIRMS_Final_For_Tableau.csv
Data quality summary: /content/drive/MyDrive/NASA_FIRMS_Tableau_Project/NASA_FIRMS_Data_Quality_Summary.csv
